In [3]:
import duckdb

In [ ]:
# Create a DuckDB connection for running SQL queries directly on the Parquet dataset.

con = duckdb.connect()

# Test that the DuckDB connection is working correctly.

con.sql("SELECT 1")

┌───────┐
│   1   │
│ int32 │
├───────┤
│     1 │
└───────┘

In [ ]:
# Define the path to the cleaned 2025 dataset.

file_path = "../Data/Processed/transportation_2025_clean.parquet"

┌─────────────────────┬───────┬───────┬───────────┬───────────────────┬─────────┬────────────────┬─────────────┬─────────┬─────────────────┬───────────┬────────────┬─────────┬──────────┬─────────────────┬──────────┬────────────┬─────────┬────────────┬─────────┬──────────┬─────────────────┬──────────┬────────────┬────────┬─────────┬──────────┬──────────────┬──────────────┬──────────┬───────────────┬───────────────────┬─────────┬─────────┐
│     FlightDate      │ Year  │ Month │ DayOfWeek │ Reporting_Airline │ Origin  │ OriginCityName │ OriginState │  Dest   │  DestCityName   │ DestState │ CRSDepTime │ DepTime │ DepDelay │ DepDelayMinutes │ DepDel15 │ DepTimeBlk │ TaxiOut │ CRSArrTime │ ArrTime │ ArrDelay │ ArrDelayMinutes │ ArrDel15 │ ArrTimeBlk │ TaxiIn │ AirTime │ Distance │ CarrierDelay │ WeatherDelay │ NASDelay │ SecurityDelay │ LateAircraftDelay │ DepHour │  Route  │
│      timestamp      │ int64 │ int64 │   int64   │      varchar      │ varchar │    varchar     │   varchar   │ var

In [ ]:
# Take a quick look at the cleaned dataset before starting the SQL analysis.

con.sql(f"""
    SELECT *
    FROM '{file_path}'
    LIMIT 5
""")

In [ ]:
# Review the key airline and route variables used throughout the analysis.

con.sql(f""" 
    SELECT
        Reporting_Airline,
        Origin,
        Dest,
        ArrDel15
    FROM '{file_path}'
    limit 5
""")

┌───────────────────┬─────────┬─────────┬──────────┐
│ Reporting_Airline │ Origin  │  Dest   │ ArrDel15 │
│      varchar      │ varchar │ varchar │  double  │
├───────────────────┼─────────┼─────────┼──────────┤
│ AA                │ JFK     │ LAX     │      0.0 │
│ AA                │ JFK     │ LAX     │      0.0 │
│ AA                │ JFK     │ LAX     │      0.0 │
│ AA                │ JFK     │ LAX     │      0.0 │
│ AA                │ JFK     │ LAX     │      0.0 │
└───────────────────┴─────────┴─────────┴──────────┘

In [ ]:
# Examine flights that experienced a 15+ minute arrival delay.

con.sql(f"""
    SELECT
        Reporting_Airline,
        Origin,
        Dest,
        ArrDel15
    FROM '{file_path}'
    WHERE ArrDel15 = 1
    LIMIT 5
""")

┌───────────────────┬─────────┬─────────┬──────────┐
│ Reporting_Airline │ Origin  │  Dest   │ ArrDel15 │
│      varchar      │ varchar │ varchar │  double  │
├───────────────────┼─────────┼─────────┼──────────┤
│ AA                │ JFK     │ LAX     │      1.0 │
│ AA                │ JFK     │ LAX     │      1.0 │
│ AA                │ LAX     │ JFK     │      1.0 │
│ AA                │ JFK     │ LAX     │      1.0 │
│ AA                │ LAX     │ JFK     │      1.0 │
└───────────────────┴─────────┴─────────┴──────────┘

In [ ]:
# Calculate the 15+ minute arrival delay rate for each airline.
# This helps compare airline-level delay performance.

con.sql(f"""
    SELECT Reporting_Airline,
        AVG(ArrDel15) AS delay_rate
    FROM '{file_path}'
    GROUP BY Reporting_Airline
    ORDER BY delay_rate DESC
""")

┌───────────────────┬─────────────────────┐
│ Reporting_Airline │     delay_rate      │
│      varchar      │       double        │
├───────────────────┼─────────────────────┤
│ F9                │ 0.27911551454230865 │
│ OH                │  0.2752976567715331 │
│ B6                │ 0.26152278531823575 │
│ AA                │ 0.25711425096107327 │
│ G4                │  0.2496535583407754 │
│ AS                │ 0.23100770067003684 │
│ YX                │  0.2151130243711139 │
│ UA                │ 0.21467438645840353 │
│ NK                │ 0.21408434497439732 │
│ WN                │ 0.21395052573398304 │
│ MQ                │ 0.21039285897853238 │
│ OO                │  0.2072171419846292 │
│ DL                │ 0.19794217262084082 │
│ HA                │ 0.17281149555681605 │
└───────────────────┴─────────────────────┘
  14 rows                       2 columns

In [ ]:
# Calculate the 15+ minute arrival delay rate for each route.
# Only routes with at least 100 flights are included to reduce the effect
# of small samples on the comparison.

con.sql(f"""
    SELECT
        Route,
        AVG(ArrDel15) AS delay_rate,
        COUNT(*) AS flight_count
    FROM '{file_path}'
    GROUP BY Route
    HAVING COUNT(*)>=100
    ORDER BY delay_rate DESC
""")

┌─────────┬──────────────────────┬──────────────┐
│  Route  │      delay_rate      │ flight_count │
│ varchar │        double        │    int64     │
├─────────┼──────────────────────┼──────────────┤
│ ROA-SFB │                0.672 │          125 │
│ TRI-SFB │   0.6422764227642277 │          123 │
│ CKB-SFB │   0.6160714285714286 │          112 │
│ LEX-SFB │   0.6136363636363636 │          220 │
│ TOL-SFB │   0.5855855855855856 │          111 │
│ SFB-ROA │                0.584 │          125 │
│ SFB-USA │   0.5808080808080808 │          198 │
│ CHA-SFB │   0.5681818181818182 │          132 │
│ SFB-GSP │   0.5663716814159292 │          113 │
│ GSP-SFB │   0.5663716814159292 │          113 │
│    ·    │            ·         │           ·  │
│    ·    │            ·         │           ·  │
│    ·    │            ·         │           ·  │
│ SJC-RNO │  0.04417670682730924 │          249 │
│ LGB-OGG │ 0.043478260869565216 │          368 │
│ SLC-EKO │ 0.041916167664670656 │          501 │


In [ ]:
# Compare airline performance using total flights, delayed flights,
# and the overall 15+ minute arrival delay rate.
# Only airlines with at least 100 flights are included.

con.sql(f"""
    SELECT
        Reporting_Airline,
        COUNT(*) AS total_flights,
        SUM(ArrDel15) AS delayed_flights,
        AVG(ArrDel15) * 100 AS delay_rate
    FROM '{file_path}'
    GROUP BY Reporting_Airline
    HAVING COUNT(*) >= 100
    ORDER BY delay_rate DESC
""")

┌───────────────────┬───────────────┬─────────────────┬────────────────────┐
│ Reporting_Airline │ total_flights │ delayed_flights │     delay_rate     │
│      varchar      │     int64     │     double      │       double       │
├───────────────────┼───────────────┼─────────────────┼────────────────────┤
│ F9                │        194192 │         54202.0 │ 27.911551454230864 │
│ OH                │        236682 │         65158.0 │  27.52976567715331 │
│ B6                │        226703 │         59288.0 │ 26.152278531823576 │
│ AA                │        952841 │        244989.0 │ 25.711425096107327 │
│ G4                │        129892 │         32428.0 │ 24.965355834077542 │
│ AS                │        241927 │         55887.0 │ 23.100770067003683 │
│ YX                │        334044 │         71857.0 │  21.51130243711139 │
│ UA                │        786377 │        168815.0 │ 21.467438645840353 │
│ NK                │        191191 │         40931.0 │ 21.408434497439732 │

In [ ]:
# Compare route performance using total flights, delayed flights,
# and the 15+ minute arrival delay rate.
# Only routes with at least 100 flights are included.

con.sql(f"""
    SELECT 
        Route,
        COUNT(*) AS total_flights,
        AVG(ArrDel15) AS delay_rate,
        SUM(ArrDel15) AS delay_flights
    FROM '{file_path}'
    GROUP BY Route 
    HAVING COUNT(*)>=100
    ORDER BY delay_rate DESC
""")

┌─────────┬───────────────┬──────────────────────┬───────────────┐
│  Route  │ total_flights │      delay_rate      │ delay_flights │
│ varchar │     int64     │        double        │    double     │
├─────────┼───────────────┼──────────────────────┼───────────────┤
│ ROA-SFB │           125 │                0.672 │          84.0 │
│ TRI-SFB │           123 │   0.6422764227642277 │          79.0 │
│ CKB-SFB │           112 │   0.6160714285714286 │          69.0 │
│ LEX-SFB │           220 │   0.6136363636363636 │         135.0 │
│ TOL-SFB │           111 │   0.5855855855855856 │          65.0 │
│ SFB-ROA │           125 │                0.584 │          73.0 │
│ SFB-USA │           198 │   0.5808080808080808 │         115.0 │
│ CHA-SFB │           132 │   0.5681818181818182 │          75.0 │
│ SFB-GSP │           113 │   0.5663716814159292 │          64.0 │
│ GSP-SFB │           113 │   0.5663716814159292 │          64.0 │
│    ·    │            ·  │            ·         │            

In [ ]:
# Calculate the total number of flights and the 15+ minute arrival delay rate
# for each month to identify seasonal differences in operational performance.

con.sql(f"""
    SELECT Month,
        COUNT(*) AS total_flights_by_month,
        AVG(ArrDel15) * 100 AS delay_rate
    FROM '{file_path}'
        GROUP BY Month
        ORDER BY Month
""")

┌───────┬────────────────────────┬────────────────────┐
│ Month │ total_flights_by_month │     delay_rate     │
│ int64 │         int64          │       double       │
├───────┼────────────────────────┼────────────────────┤
│     1 │                 522269 │  18.78916803409737 │
│     2 │                 496476 │  20.76676415375567 │
│     3 │                 592301 │ 19.590377189976042 │
│     4 │                 577730 │ 19.663856818929258 │
│     5 │                 597574 │ 23.601763128917923 │
│     6 │                 599472 │ 28.259034617129743 │
│     7 │                 612811 │  28.88525173340557 │
│     8 │                 593733 │ 22.555593170667624 │
│     9 │                 558329 │ 16.631621555787994 │
│    10 │                 601570 │ 20.321990790764165 │
│    11 │                 555296 │ 20.560205728116177 │
│    12 │                 571924 │ 26.771564053965214 │
└───────┴────────────────────────┴────────────────────┘
  12 rows                                   3 co

In [ ]:
# Count the number of 15+ minute arrival delays for each airline.
# This measures the absolute number of delayed flights rather than the delay rate.

con.sql(f"""
    SELECT Reporting_Airline,
        COUNT(*) AS number_of_delay_flights
    FROM '{file_path}'
    WHERE ArrDel15 = 1
    GROUP BY Reporting_Airline
    ORDER BY number_of_delay_flights DESC
""")

┌───────────────────┬─────────────────────────┐
│ Reporting_Airline │ number_of_delay_flights │
│      varchar      │          int64          │
├───────────────────┼─────────────────────────┤
│ B6                │                   59288 │
│ MQ                │                   61400 │
│ OH                │                   65158 │
│ AA                │                  244989 │
│ HA                │                   13710 │
│ G4                │                   32428 │
│ UA                │                  168815 │
│ OO                │                  170996 │
│ YX                │                   71857 │
│ AS                │                   55887 │
│ DL                │                  200402 │
│ F9                │                   54202 │
│ WN                │                  294575 │
│ NK                │                   40931 │
└───────────────────┴─────────────────────────┘
  14 rows                           2 columns

In [ ]:
# Calculate the 15+ minute arrival delay rate by scheduled departure hour.
# This helps identify time periods when operations are more vulnerable to delays.

con.sql(f"""
    SELECT DepHour,
        AVG(ArrDel15) * 100 AS delay_rate
    FROM '{file_path}'
        GROUP BY DepHour
        ORDER BY delay_rate DESC
""")


┌─────────┬────────────────────┐
│ DepHour │     delay_rate     │
│  int64  │       double       │
├─────────┼────────────────────┤
│      24 │              100.0 │
│      19 │  32.57949810961046 │
│      20 │  31.77828749699708 │
│      18 │ 31.765724177726483 │
│      17 │  31.24878563299747 │
│      16 │  29.65631100903725 │
│      21 │ 29.067454258317266 │
│      22 │ 28.844897852552382 │
│      15 │ 27.916003109120506 │
│      14 │  25.77087966830034 │
│       · │          ·         │
│       · │          ·         │
│       · │          ·         │
│       3 │  17.89617486338798 │
│      10 │ 17.257038842901732 │
│       1 │ 17.152961980548188 │
│       0 │  16.54176072234763 │
│       9 │ 15.827700843785749 │
│       8 │ 14.468719527091785 │
│       4 │ 13.422818791946309 │
│       7 │ 12.731411743815551 │
│       6 │ 10.056371970743875 │
│       5 │  8.851850141845063 │
└─────────┴────────────────────┘
  25 rows (20 shown) 2 columns

In [ ]:
# The original data contains a single 24:00 value.
# Convert 24 to 0 so midnight is treated consistently with other 0-hour values.

con.sql(f"""
    SELECT
        CASE WHEN DepHour = 24 THEN 0 ELSE DepHour END AS DepHour,
        AVG(ArrDel15) * 100 AS delay_rate
    FROM '{file_path}'
    GROUP BY 1
    ORDER BY delay_rate DESC
""")

┌─────────┬────────────────────┐
│ DepHour │     delay_rate     │
│  int64  │       double       │
├─────────┼────────────────────┤
│      19 │  32.57949810961046 │
│      20 │  31.77828749699708 │
│      18 │ 31.765724177726483 │
│      17 │  31.24878563299747 │
│      16 │  29.65631100903725 │
│      21 │ 29.067454258317266 │
│      22 │ 28.844897852552382 │
│      15 │ 27.916003109120506 │
│      14 │  25.77087966830034 │
│      13 │   22.7044547022939 │
│       · │           ·        │
│       · │           ·        │
│       · │           ·        │
│       3 │  17.89617486338798 │
│      10 │ 17.257038842901732 │
│       1 │ 17.152961980548188 │
│       0 │ 16.549295774647888 │
│       9 │ 15.827700843785749 │
│       8 │ 14.468719527091785 │
│       4 │ 13.422818791946309 │
│       7 │ 12.731411743815551 │
│       6 │ 10.056371970743875 │
│       5 │  8.851850141845063 │
└─────────┴────────────────────┘
  24 rows (20 shown) 2 columns

In [ ]:
# Calculate the total delay minutes contributed by each major delay cause.
# This helps identify which causes have the greatest overall operational impact.

con.sql(f"""
    SELECT
        SUM(CarrierDelay) AS carrier_delay_minutes,
        SUM(WeatherDelay) AS weather_delay_minutes,
        SUM(NASDelay) AS nas_delay_minutes,
        SUM(SecurityDelay) AS security_delay_minutes,
        SUM(LateAircraftDelay) AS late_aircraft_delay_minutes
    FROM '{file_path}'
""")

┌───────────────────────┬───────────────────────┬───────────────────┬────────────────────────┬─────────────────────────────┐
│ carrier_delay_minutes │ weather_delay_minutes │ nas_delay_minutes │ security_delay_minutes │ late_aircraft_delay_minutes │
│        double         │        double         │      double       │         double         │           double            │
├───────────────────────┼───────────────────────┼───────────────────┼────────────────────────┼─────────────────────────────┤
│            36326377.0 │             7108714.0 │        24483761.0 │               152802.0 │                  43862971.0 │
└───────────────────────┴───────────────────────┴───────────────────┴────────────────────────┴─────────────────────────────┘

In [ ]:
# Identify airline and origin airport combinations with higher
# 15+ minute arrival delay rates.
# Only combinations with at least 100 flights are included
# to reduce the effect of small samples.

con.sql(f"""
    SELECT Reporting_Airline,
        Origin,
        AVG(ArrDel15) * 100 AS delay_rate
    FROM '{file_path}'
    GROUP BY Reporting_Airline, Origin
    HAVING COUNT(*) >= 100
    ORDER BY delay_rate DESC
""")


┌───────────────────┬─────────┬────────────────────┐
│ Reporting_Airline │ Origin  │     delay_rate     │
│      varchar      │ varchar │       double       │
├───────────────────┼─────────┼────────────────────┤
│ HA                │ ANC     │  69.18238993710692 │
│ G4                │ TRI     │               62.5 │
│ G4                │ LIT     │ 54.347826086956516 │
│ G4                │ CKB     │  51.75879396984925 │
│ G4                │ HGR     │  49.67532467532468 │
│ B6                │ MKE     │ 49.411764705882355 │
│ G4                │ ROA     │ 49.279538904899134 │
│ G4                │ OKC     │               48.8 │
│ G4                │ ORF     │ 45.604395604395606 │
│ G4                │ USA     │  45.23470839260313 │
│ ·                 │  ·      │          ·         │
│ ·                 │  ·      │          ·         │
│ ·                 │  ·      │          ·         │
│ OO                │ PIH     │  5.646630236794172 │
│ MQ                │ LAX     │            5.4